In [ ]:
# Test Notebook for TroposphereLightningModule

import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
import lightning as L

import sys
sys.path.insert(0, r'C:\Users\micke\OneDrive - University of Toronto\geo_lab')
from geolab.data.datamodule.troposphere_datamodule import TroposphereDataModule
from geolab.models.modules.troposphere_module import TroposphereLightningModule

# ============================================================================
# CONFIGURATION - UPDATE THESE
# ============================================================================
DATA_DIR = Path(r"C:\Users\micke\Downloads\dfda178b22d3772b9b9b0118dba68a02.nc")  # UPDATE THIS
SOLUTION_VARS = ['w', 'u', 'z', 'v']  # UPDATE WITH YOUR VARIABLES

# ============================================================================
# Setup: Create DataModule First
# ============================================================================
def setup_datamodule():
    """Create and setup datamodule for testing."""
    print("=" * 60)
    print("Setting up DataModule")
    print("=" * 60)

    datamodule = TroposphereDataModule(
        data_dir=DATA_DIR,
        solution_vars=SOLUTION_VARS,
        val_split=0.15,
        test_split=0.15,
        batch_size=32,
        num_workers=2,
        seed=42
    )

    datamodule.setup(stage='fit')

    print(f"\nDataModule ready!")
    print(f"  Train samples: {datamodule.num_train_samples:,}")
    print(f"  Val samples: {datamodule.num_val_samples:,}")
    print(f"  Input dim: {datamodule.input_dim}")
    print(f"  Output dim: {datamodule.output_dim}")

    return datamodule


# ============================================================================
# Test 1: Initialize Lightning Module (Standard MLP)
# ============================================================================
def test_init_standard_mlp(datamodule):
    """Test initializing a standard MLP without PINN."""
    print("\n" + "=" * 60)
    print("Test 1: Initialize Standard MLP")
    print("=" * 60)

    model = TroposphereLightningModule(
        model_name='FCN',
        N_hidden_features=128,
        N_hidden_layers=4,
        model_params={
            'activation': 'relu',
            'bias': True,
            'init_type': 'uniform'
        },
        position_encoder_type=None,  # No positional encoding
        mapping_dim=None,
        scale=None,
        optimizer_name='Adam',
        optimizer_config={'lr': 1e-3},
        scheduler_name=None,
        train_pinn=False,
        datamodule=datamodule
    )

    print(f"\nModel initialized successfully!")
    print(f"  Model name: {model.hparams.model_name}")
    print(f"  Input features: {model.N_in_features}")
    print(f"  Output features: {model.N_out_features}")
    print(f"  Hidden features: {model.hparams.N_hidden_features}")
    print(f"  Hidden layers: {model.hparams.N_hidden_layers}")
    print(f"  PINN training: {model.train_pinn}")

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"\n  Total parameters: {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,}")

    return model


# ============================================================================
# Test 2: Initialize with Fourier Features
# ============================================================================
def test_init_with_fourier(datamodule):
    """Test initializing model with Fourier feature encoding."""
    print("\n" + "=" * 60)
    print("Test 2: Initialize with Fourier Features")
    print("=" * 60)

    model = TroposphereLightningModule(
        model_name='FCN',
        N_hidden_features=256,
        N_hidden_layers=6,
        model_params={
            'activation': 'relu',
            'bias': True,
            'init_type': 'uniform'
        },
        position_encoder_type='gaussian',
        mapping_dim=256,
        scale=10.0,
        encode_coords=['latitude', 'longitude'],  # Encode spatial coords only
        optimizer_name='Adam',
        optimizer_config={'lr': 1e-3},
        train_pinn=False,
        datamodule=datamodule
    )

    print(f"\nModel with Fourier features initialized!")
    print(f"  Position encoder: {model.hparams.position_encoder_type}")
    print(f"  Mapping dimension: {model.hparams.mapping_dim}")
    print(f"  Scale: {model.hparams.scale}")

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    print(f"  Total parameters: {total_params:,}")

    return model


# ============================================================================
# Test 3: Initialize PINN Model
# ============================================================================
def test_init_pinn(datamodule):
    """Test initializing a PINN model."""
    print("\n" + "=" * 60)
    print("Test 3: Initialize PINN Model")
    print("=" * 60)

    model = TroposphereLightningModule(
        model_name='FCN',
        N_hidden_features=128,
        N_hidden_layers=4,
        model_params={
            'activation': 'relu',
            'bias': True,
            'init_type': 'uniform'
        },
        position_encoder_type=None,
        mapping_dim=None,
        scale=None,
        optimizer_name='Adam',
        optimizer_config={'lr': 1e-3},
        train_pinn=True,  # Enable PINN
        mass_balance=True,
        physics_loss_weight=0.5,
        include_virtual=True,
        num_virtual_per_batch=1000,
        datamodule=datamodule
    )

    print(f"\nPINN model initialized!")
    print(f"  PINN training: {model.train_pinn}")
    print(f"  Mass balance: {model.mass_balance}")
    print(f"  Physics loss weight: {model.physics_loss_weight}")
    print(f"  Include virtual points: {model.include_virtual}")
    print(f"  Virtual points per batch: {model.num_virtual_per_batch}")

    return model


# ============================================================================
# Test 4: Forward Pass
# ============================================================================
def test_forward_pass(model, datamodule):
    """Test forward pass through the model."""
    print("\n" + "=" * 60)
    print("Test 4: Forward Pass")
    print("=" * 60)

    # Get a batch
    train_loader = datamodule.train_dataloader()
    batch = next(iter(train_loader))

    # Normalize
    batch_norm = datamodule.normalize_batch(batch)

    # Forward pass
    coords = batch_norm['coords']
    preds = model(coords)

    print(f"\n  Input shape: {coords.shape}")
    print(f"  Output shape: {preds.shape}")
    print(f"  Output range: [{preds.min():.4f}, {preds.max():.4f}]")

    # Check dimensions
    assert preds.shape[0] == coords.shape[0], "Batch size mismatch"
    assert preds.shape[1] == datamodule.output_dim, "Output dimension mismatch"

    print(f"\n  PASS: Forward pass successful!")

    return preds


# ============================================================================
# Test 5: Training Step (Standard)
# ============================================================================
def test_training_step_standard(model, datamodule):
    """Test training step for standard model."""
    print("\n" + "=" * 60)
    print("Test 5: Training Step (Standard)")
    print("=" * 60)

    # Get a batch
    train_loader = datamodule.train_dataloader()
    batch = next(iter(train_loader))

    # Training step
    model.train()
    loss = model.training_step(batch, batch_idx=0)

    print(f"\n  Loss: {loss.item():.6f}")
    print(f"  Loss requires grad: {loss.requires_grad}")

    # Check gradients were computed
    has_grads = any(p.grad is not None for p in model.parameters())
    if has_grads:
        print(f"  WARNING: Gradients computed (expected in full training loop)")
    else:
        print(f"  No gradients yet (normal for single step)")

    print(f"\n  PASS: Training step executed!")

    return loss


# ============================================================================
# Test 6: Training Step (PINN)
# ============================================================================
def test_training_step_pinn(model_pinn, datamodule):
    """Test training step for PINN model."""
    print("\n" + "=" * 60)
    print("Test 6: Training Step (PINN)")
    print("=" * 60)

    # Get a batch
    train_loader = datamodule.train_dataloader()
    batch = next(iter(train_loader))

    print(f"\n  Original batch size: {batch['coords'].shape[0]}")

    # Training step (will augment with virtual points)
    model_pinn.train()
    loss = model_pinn.training_step(batch, batch_idx=0)

    print(f"\n  Total loss: {loss.item():.6f}")

    # Check metrics were updated
    print(f"\n  Logged metrics:")
    print(f"    Data loss: {model_pinn.train_data_loss.compute():.6f}")
    print(f"    Physics loss: {model_pinn.train_physics_loss.compute():.6f}")
    print(f"    Mass continuity: {model_pinn.train_mass_cont.compute():.6f}")

    print(f"\n  PASS: PINN training step executed!")

    return loss


# ============================================================================
# Test 7: Validation Step
# ============================================================================
def test_validation_step(model, datamodule):
    """Test validation step."""
    print("\n" + "=" * 60)
    print("Test 7: Validation Step")
    print("=" * 60)

    # Get a validation batch
    val_loader = datamodule.val_dataloader()
    batch = next(iter(val_loader))

    # Validation step
    model.eval()
    with torch.no_grad():
        loss = model.validation_step(batch, batch_idx=0)

    print(f"\n  Validation loss: {loss.item():.6f}")
    print(f"\n  PASS: Validation step executed!")

    return loss


# ============================================================================
# Test 8: Virtual Point Generation
# ============================================================================
def test_virtual_points(model_pinn):
    """Test virtual point generation for PINN."""
    print("\n" + "=" * 60)
    print("Test 8: Virtual Point Generation")
    print("=" * 60)

    n_virtual = 1000
    virtual_coords = model_pinn.generate_virtual_points(n_virtual)

    print(f"\n  Generated {n_virtual} virtual points")
    print(f"  Virtual coords shape: {virtual_coords.shape}")
    print(f"  Virtual coords range: [{virtual_coords.min():.4f}, {virtual_coords.max():.4f}]")

    # Visualize distribution in 2D projections
    coords_np = virtual_coords.cpu().numpy()

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    coord_pairs = [
        (0, 1, 'Time', 'Pressure'),
        (0, 2, 'Time', 'Latitude'),
        (0, 3, 'Time', 'Longitude'),
        (1, 2, 'Pressure', 'Latitude'),
        (1, 3, 'Pressure', 'Longitude'),
        (2, 3, 'Latitude', 'Longitude')
    ]

    for ax, (i, j, label_i, label_j) in zip(axes, coord_pairs):
        ax.scatter(coords_np[:, i], coords_np[:, j], alpha=0.3, s=5)
        ax.set_xlabel(label_i, fontsize=10)
        ax.set_ylabel(label_j, fontsize=10)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.suptitle('Virtual Point Distribution (LHS Sampling)', y=1.01, fontsize=14)
    plt.show()

    print(f"\n  PASS: Virtual points generated and visualized!")


# ============================================================================
# Test 9: Batch Augmentation
# ============================================================================
def test_batch_augmentation(model_pinn, datamodule):
    """Test batch augmentation with virtual points."""
    print("\n" + "=" * 60)
    print("Test 9: Batch Augmentation")
    print("=" * 60)

    # Get a batch
    train_loader = datamodule.train_dataloader()
    batch = next(iter(train_loader))
    batch = datamodule.normalize_batch(batch)

    original_size = batch['coords'].shape[0]
    print(f"\n  Original batch size: {original_size}")

    # Augment
    augmented = model_pinn.augment_batch_with_virtual(batch)

    augmented_size = augmented['coords'].shape[0]
    n_virtual = model_pinn.num_virtual_per_batch

    print(f"  Virtual points added: {n_virtual}")
    print(f"  Augmented batch size: {augmented_size}")
    print(f"  Expected size: {original_size + n_virtual}")

    # Check classification mask
    n_real = augmented['classification'].sum().item()
    n_virtual_mask = (~augmented['classification']).sum().item()

    print(f"\n  Classification mask:")
    print(f"    Real points: {n_real}")
    print(f"    Virtual points: {n_virtual_mask}")

    assert n_real == original_size, "Real point count mismatch"
    assert n_virtual_mask == n_virtual, "Virtual point count mismatch"

    print(f"\n  PASS: Batch augmentation successful!")


# ============================================================================
# Test 10: Evaluation on Grid
# ============================================================================
def test_evaluate_on_grid(model, datamodule):
    """Test evaluation on a regular grid."""
    print("\n" + "=" * 60)
    print("Test 10: Evaluate on Grid")
    print("=" * 60)

    # Create small test grid
    from geolab.viz import GeometryGenerator

    geometry = GeometryGenerator(
        coord_domain=datamodule.coordinate_ranges,
        coord_labels=datamodule.data.coord_labels,
        resolution={'longitude': 20.0, 'latitude': 10.0}
    )

    # Create horizontal slice
    coords = geometry.plane(
        axes=['longitude', 'latitude'],
        pressure_level=500,
        valid_time=datamodule.coordinate_ranges['valid_time'][0]
    )

    print(f"\n  Grid shape: {coords.shape}")

    # Evaluate
    model.eval()
    preds = model.evaluate_on_grid(coords, denormalize=True)

    print(f"  Predictions shape: {preds.shape}")
    print(f"  Predictions range: [{preds.min():.4f}, {preds.max():.4f}]")

    # Visualize first variable
    n_lon = geometry.compute_space_size(['longitude', 'latitude'])['longitude']
    n_lat = geometry.compute_space_size(['longitude', 'latitude'])['latitude']

    pred_field = preds[:, 0].reshape(n_lon, n_lat).cpu().numpy()
    lon_grid = coords[:, 0].reshape(n_lon, n_lat).numpy()
    lat_grid = coords[:, 1].reshape(n_lon, n_lat).numpy()

    fig, ax = plt.subplots(figsize=(12, 6))
    im = ax.contourf(lon_grid, lat_grid, pred_field, levels=20, cmap='RdBu_r')
    ax.set_xlabel('Longitude (°)', fontsize=12)
    ax.set_ylabel('Latitude (°)', fontsize=12)
    ax.set_title(f'{SOLUTION_VARS[0].upper()} Prediction at 500 hPa', fontsize=14)
    plt.colorbar(im, ax=ax, label=SOLUTION_VARS[0])
    plt.tight_layout()
    plt.show()

    print(f"\n  PASS: Grid evaluation successful!")


# ============================================================================
# Test 11: Optimizer Configuration
# ============================================================================
def test_optimizer_config(datamodule):
    """Test different optimizer configurations."""
    print("\n" + "=" * 60)
    print("Test 11: Optimizer Configuration")
    print("=" * 60)

    # Test with scheduler
    model = TroposphereLightningModule(
        model_name='FCN',
        N_hidden_features=64,
        N_hidden_layers=2,
        model_params={'activation': 'relu'},
        position_encoder_type=None,
        mapping_dim=None,
        scale=None,
        optimizer_name='AdamW',
        optimizer_config={'lr': 1e-3, 'weight_decay': 1e-4},
        scheduler_name='CosineAnnealingLR',
        scheduler_config={'T_max': 100},
        train_pinn=False,
        datamodule=datamodule
    )

    # Configure optimizer
    optimizer = model.configure_optimizers()

    if isinstance(optimizer, tuple):
        optimizers, schedulers = optimizer
        print(f"\n  Optimizer: {optimizers[0].__class__.__name__}")
        print(f"  Scheduler: {schedulers[0].__class__.__name__}")
    else:
        print(f"\n  Optimizer: {optimizer.__class__.__name__}")
        print(f"  No scheduler configured")

    print(f"\n  PASS: Optimizer configured successfully!")


# ============================================================================
# Test 12: Statistics Format Conversion
# ============================================================================
def test_statistics_conversion(model, datamodule):
    """Test statistics format conversion for physics code."""
    print("\n" + "=" * 60)
    print("Test 12: Statistics Format Conversion")
    print("=" * 60)

    # Get converted statistics
    stats_physics = model._convert_statistics_format()

    print(f"\n  Converted statistics:")
    print(f"  Variables:")
    for var in SOLUTION_VARS:
        print(f"    {var}: {stats_physics[var]}")

    print(f"\n  Coordinates:")
    for coord in ['valid_time', 'pressure_level', 'latitude', 'longitude']:
        print(f"    {coord}: {stats_physics[coord]}")

    print(f"\n  PASS: Statistics conversion successful!")


# ============================================================================
# Run All Tests
# ============================================================================
def run_all_tests():
    """Run all TroposphereLightningModule tests."""
    print("\n" + "=" * 70)
    print(" TroposphereLightningModule Test Suite")
    print("=" * 70)

    # Setup datamodule
    datamodule = setup_datamodule()

    # Test 1-3: Different initializations
    model_standard = test_init_standard_mlp(datamodule)
    model_fourier = test_init_with_fourier(datamodule)
    model_pinn = test_init_pinn(datamodule)

    # Test 4-5: Forward and training (standard)
    preds = test_forward_pass(model_standard, datamodule)
    loss_standard = test_training_step_standard(model_standard, datamodule)

    # Test 6: Training (PINN)
    loss_pinn = test_training_step_pinn(model_pinn, datamodule)

    # Test 7: Validation
    val_loss = test_validation_step(model_standard, datamodule)

    # Test 8-9: PINN-specific
    test_virtual_points(model_pinn)
    test_batch_augmentation(model_pinn, datamodule)

    # Test 10: Evaluation
    test_evaluate_on_grid(model_standard, datamodule)

    # Test 11-12: Configuration
    test_optimizer_config(datamodule)
    test_statistics_conversion(model_standard, datamodule)

    print("\n" + "=" * 70)
    print(" All Tests Completed Successfully!")
    print("=" * 70)

    return {
        'datamodule': datamodule,
        'model_standard': model_standard,
        'model_fourier': model_fourier,
        'model_pinn': model_pinn
    }


# ============================================================================
# Run Tests
# ============================================================================
if __name__ == "__main__":
    results = run_all_tests()


 TroposphereLightningModule Test Suite
Setting up DataModule
Loading ERA5 data from C:\Users\micke\Downloads\dfda178b22d3772b9b9b0118dba68a02.nc
No slicing applied - using all indices
Coordinate ranges:
  valid_time: shape=(24,), range=[1664323200.00, 1664406016.00]
  pressure_level: shape=(17,), range=[200.00, 850.00]
  latitude: shape=(721,), range=[-90.00, 90.00]
  longitude: shape=(1440,), range=[-180.00, 179.75]
Preloading data into memory...
  Loading w...
    Memory usage: 1.69 GB
  Loading u...
    Memory usage: 1.69 GB
  Loading z...
    Memory usage: 1.69 GB
  Loading v...
    Memory usage: 1.69 GB
Successfully loaded 4 variables:
   w: (24, 17, 721, 1440)
   u: (24, 17, 721, 1440)
   z: (24, 17, 721, 1440)
   v: (24, 17, 721, 1440)
Total 4D grid points: 423,601,920
  Shape: (time=24, pressure=17, lat=721, lon=1440)
Split complete:
  Train: 296,521,344 points (70.0%)
  Val:   63,540,288 points (15.0%)
  Test:  63,540,288 points (15.0%)
Loading statistics from C:\Users\micke\